In [0]:
account_key = dbutils.secrets.get(scope='databricks-scope' ,key = 'databricks-strg-access-key' )

In [0]:
spark.conf.set("fs.azure.account.key.databricksrg2026.dfs.core.windows.net",account_key)

In [0]:
races_df = spark.read.parquet("abfss://processed@databricksrg2026.dfs.core.windows.net/races")\
    .withColumnRenamed("name", "race_name")\
        .withColumnRenamed("round","race_round")

In [0]:
constructor_df = spark.read.parquet("abfss://processed@databricksrg2026.dfs.core.windows.net/constructor")\
.withColumnRenamed("name", "constructorName")

In [0]:
circuit_df = spark.read.csv("abfss://raw@databricksrg2026.dfs.core.windows.net/circuits.csv",\
    inferSchema=True, 
    header=True,)\
    .withColumnRenamed("name", "circuit_name")\
        .withColumnRenamed("location", "circuit_location")
    

In [0]:
drivers_df = spark.read.parquet("abfss://processed@databricksrg2026.dfs.core.windows.net/drivers")\
    .withColumnRenamed("name", "driver_name")\
        .withColumnRenamed("dob", "driver_dob")\
            .withColumnRenamed("nationality", "driver_nationality")\
                .withColumnRenamed("code", "driver_code")\
                    .withColumnRenamed("number", "driver_number")


In [0]:
results_df = spark.read.parquet("abfss://processed@databricksrg2026.dfs.core.windows.net/results")\
    .withColumnRenamed("time", "race_time")


Join circuit to **races**

In [0]:
races_circuit_df = races_df.join(circuit_df, races_df.circuit_id == circuit_df.circuitId, "inner")\
  .select(races_df.race_id , races_df.race_name, races_df.race_year , circuit_df.circuit_location)
display(races_circuit_df)


In [0]:
race_results_df = results_df.join(races_circuit_df , results_df.race_id == races_circuit_df .race_id,"inner")\
    .join(drivers_df, results_df.driver_id == drivers_df.driver_id, "inner")\
        .join(constructor_df, results_df.constructor_id == constructor_df.constructor_id, "inner")\
            

In [0]:
from pyspark.sql.functions import current_timestamp

In [0]:
    final_df = race_results_df.select("race_year", "race_name", "driver_name", "driver_number", "driver_nationality", "constructorName", "circuit_location", "race_time","fastest_lap","points","position")\
        .withColumn("created_date", current_timestamp())
    
    
    


In [0]:
display(final_df)
final_df.write.mode("overwrite").parquet("abfss://presentation@databricksrg2026.dfs.core.windows.net/")